In [ ]:
import struct
from pathlib import Path


def find_file(root, *names):
    root = Path(root)
    for name in names:
        for file_path in root.rglob(name):
            if file_path.is_file():
                return file_path
    raise FileNotFoundError(f"Не найден файл {names} в {root}")


def read_idx_images(file_path):
    import numpy as np

    with open(file_path, "rb") as file:
        magic, count, rows, columns = struct.unpack(">IIII", file.read(16))
        if magic != 2051:
            raise ValueError(f"Неверный IDX-файл изображений: {file_path}")
        images = np.frombuffer(file.read(), dtype=np.uint8)
    return images.reshape(count, rows, columns)


def read_idx_labels(file_path):
    import numpy as np

    with open(file_path, "rb") as file:
        magic, count = struct.unpack(">II", file.read(8))
        if magic != 2049:
            raise ValueError(f"Неверный IDX-файл меток: {file_path}")
        labels = np.frombuffer(file.read(), dtype=np.uint8)
    if len(labels) != count:
        raise ValueError(f"Число меток не совпадает с заголовком файла: {file_path}")
    return labels


def main():
    try:
        import kagglehub
        import matplotlib.pyplot as plt
        import numpy as np
        import seaborn as sns
        import tensorflow as tf
        from sklearn.metrics import classification_report, confusion_matrix
        from sklearn.model_selection import train_test_split
    except ImportError as error:
        raise SystemExit(
            "Установите зависимости: pip install kagglehub numpy matplotlib seaborn scikit-learn tensorflow"
        ) from error

    seed = 42
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print("TensorFlow:", tf.__version__)


    cache_root = Path.home() / ".cache" / "kagglehub" / "datasets" / "hojjatk" / "mnist-dataset" / "versions" / "1"
    if (cache_root / "train-images.idx3-ubyte").is_file():
        root = cache_root
    else:
        root = Path(kagglehub.dataset_download("hojjatk/mnist-dataset"))
    print("Путь к датасету:", root)

    x_train_full = read_idx_images(
        find_file(root, "train-images-idx3-ubyte", "train-images.idx3-ubyte")
    )
    y_train_full = read_idx_labels(
        find_file(root, "train-labels-idx1-ubyte", "train-labels.idx1-ubyte")
    )
    x_test = read_idx_images(
        find_file(root, "t10k-images-idx3-ubyte", "t10k-images.idx3-ubyte")
    )
    y_test = read_idx_labels(
        find_file(root, "t10k-labels-idx1-ubyte", "t10k-labels.idx1-ubyte")
    )


    x_train, x_val, y_train, y_val = train_test_split(
        x_train_full,
        y_train_full,
        test_size=0.1,
        random_state=seed,
        stratify=y_train_full,
    )
    x_train = x_train.astype("float32") / 255.0
    x_val = x_val.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    print(f"Train:      X={x_train.shape}, y={y_train.shape}")
    print(f"Validation: X={x_val.shape}, y={y_val.shape}")
    print(f"Test:       X={x_test.shape}, y={y_test.shape}")


    print("\nКоличество примеров каждого класса в train:")
    for digit, count in zip(*np.unique(y_train, return_counts=True)):
        print(f"Цифра {digit}: {count}")

    fig, axes = plt.subplots(2, 10, figsize=(15, 4))
    for digit in range(10):
        indices = np.where(y_train == digit)[0][:2]
        for row, index in enumerate(indices):
            axes[row, digit].imshow(x_train[index], cmap="gray")
            axes[row, digit].set_title(f"Метка: {digit}")
            axes[row, digit].axis("off")
    plt.suptitle("Примеры изображений MNIST")
    plt.tight_layout()
    plt.show()


    x_train_cnn = x_train[..., np.newaxis]
    x_val_cnn = x_val[..., np.newaxis]
    x_test_cnn = x_test[..., np.newaxis]

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, kernel_size=3, activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(64, kernel_size=3, activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    model.summary()

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True
    )
    history = model.fit(
        x_train_cnn,
        y_train,
        validation_data=(x_val_cnn, y_val),
        epochs=4,
        batch_size=128,
        callbacks=[early_stopping],
    )


    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["accuracy"], label="Train")
    axes[0].plot(history.history["val_accuracy"], label="Validation")
    axes[0].set_title("Accuracy по эпохам")
    axes[0].set_xlabel("Эпоха")
    axes[0].legend()
    axes[1].plot(history.history["loss"], label="Train")
    axes[1].plot(history.history["val_loss"], label="Validation")
    axes[1].set_title("Loss по эпохам")
    axes[1].set_xlabel("Эпоха")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


    test_loss, test_accuracy = model.evaluate(x_test_cnn, y_test, verbose=0)
    probabilities = model.predict(x_test_cnn, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    print(f"\nTest loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_accuracy:.4f}")
    print(classification_report(y_test, predictions, digits=4))

    matrix = confusion_matrix(y_test, predictions)
    plt.figure(figsize=(9, 7))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Предсказанная цифра")
    plt.ylabel("Истинная цифра")
    plt.title("Матрица ошибок на тестовой выборке")
    plt.tight_layout()
    plt.show()

    wrong = np.where(predictions != y_test)[0]
    indices = wrong[:10] if len(wrong) else np.arange(min(10, len(y_test)))
    print("Ошибочных предсказаний:", len(wrong), "из", len(y_test))
    fig, axes = plt.subplots(1, len(indices), figsize=(15, 2.5))
    if len(indices) == 1:
        axes = [axes]
    for ax, index in zip(axes, indices):
        ax.imshow(x_test[index], cmap="gray")
        ax.set_title(f"Истина: {y_test[index]}\nПрогноз: {predictions[index]}")
        ax.axis("off")
    plt.suptitle("Примеры ошибок" if len(wrong) else "Примеры верных прогнозов")
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()


- На test из 10 000 изображений CNN получила `accuracy=0.9899` (`98.99%`), `loss=0.0317`; неверно классифицировано 101 изображение.
- Для слоя свёртки стоимость одного прохода — `O(B·H·W·Cᵢₙ·Cₒᵤₜ·K²)`. Для всей сети обучение пропорционально числу эпох и изображений: `O(E·N·Σ(H·W·Cᵢₙ·Cₒᵤₜ·K²))`. Память включает веса, активации пакета и загруженные изображения.